# Tensile Test (TTO): from data entry to RDF

This notebook shows how to describe a uniaxial tensile test and its measured
results, and convert that description into a standardised, machine-readable
RDF graph using the
[Tensile Test Ontology (TTO)](https://w3id.org/pmd/tto/) and the
[Platform MaterialDigital Core Ontology (PMDCo)](https://w3id.org/pmd/co/).

**You only need to edit one file:** `docs/example.input.json`. Everything
else is automatic.


## How this schema relates to the characterization base

This schema **extends** `characterization/step/PMDCo/` via JSON Schema `$ref`
and `allOf`. Think of it as inheritance: the tensile test schema reuses all
the structural fields of the generic characterization step (specimen input,
test conditions, process chain position) and adds its own result fields on top.

```
characterization/step/PMDCo/          ← base schema
  has_specified_input  (specimen IRI)
  preceded_by          (chain)
  has_process_condition (conditions)

characterization/tensile-test/TTO/    ← this schema (extends base)
  type: tto:TensileTest               overrides root class
  measured_properties                 adds result nodes (TTO classes)
```


## What the notebook does

```
example.input.json
  │  specimen IRI, test conditions, measured properties
  │
  ▼
Transform
  │  converts plain JSON to a structured OO-LD document
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
SHACL validation  →  confirms the graph is structurally correct
SPARQL inspect    →  shows the test results captured in the graph
```


## Environment setup

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install jupyterlab
jupyter lab
```

In [ ]:
%pip install -q jsonata-python rdflib pyshacl pyyaml

In [ ]:
import sys, json, pathlib, yaml, pyshacl, rdflib
from jsonata.jsonata import Jsonata
from importlib.metadata import version

print("Python        :", sys.executable)
print("jsonata-python", version("jsonata-python"))
print("rdflib        ", version("rdflib"))
print("pyshacl       ", version("pyshacl"))

HERE       = pathlib.Path().resolve()          # docs/
SCHEMA     = HERE.parent                       # tensile-test/TTO/
STORE_ROOT = SCHEMA.parents[3]                 # semantic-schemas/
CHAR_BASE  = STORE_ROOT / "schemas" / "characterization" / "step" / "PMDCo"

## Step 1: Describe your tensile test

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `test_name` | yes | A name for this test |
| `specimen_iri` | yes | IRI of the specimen tested (must exist in the knowledge graph) |
| `test_standard` | no | Standard followed, e.g. `"ISO 6892-1"` |
| `strain_rate` | no | Strain or crosshead displacement rate |
| `strain_rate_unit` | no | Unit for strain rate; defaults to `"1/s"` |
| `temperature` | no | Test temperature in °C |
| `results` | no | List of measured properties: `property`, `value`, `unit` |

Supported property names: `YieldStrength`, `TensileStrength`,
`PercentageElongationAfterFracture`, `PercentageReductionOfArea`,
`UpperYieldStrength`, `LowerYieldStrength`, `ProofStrength`,
`PercentagePermanentElongation`.

Supported units: `"MPa"`, `"GPa"` for stress properties; `"%"` for
elongation and reduction of area.

In [ ]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

## Step 2: Convert to OO-LD

The transform converts your plain input into a structured OO-LD document.
Because this schema **extends** the characterization base, the converter
produces a single merged document that contains both the inherited fields
(specimen input, test conditions) and the new tensile-test-specific fields
(measured properties typed to TTO classes).

Each result entry becomes a node typed to the corresponding TTO class
(e.g. `tto:YieldStrength`), with a numeric value and a QUDT unit IRI.

In [ ]:
expr     = (SCHEMA / "simplified" / "transform.jsonata").read_text()
oold_doc = Jsonata(expr).evaluate(simplified)

print(json.dumps(oold_doc, indent=2))

## Step 3: Convert to RDF

The OO-LD document is parsed into an RDF graph using the ontology context
from `specs/schema.oold.yaml`. The context maps every field to its precise
ontology IRI, for example:

| JSON field | Ontology IRI |
|---|---|
| `type` | `rdf:type` |
| `has_specified_input` | `OBI_0000293` |
| `measured_properties` | `OBI_0000299` (has_specified_output) |
| `result_value` | `OBI_0001937` (has_specified_numeric_value) |
| `result_unit` | `IAO_0000039` (has_measurement_unit_label) |

In [ ]:
context = yaml.safe_load((SCHEMA / "specs" / "schema.oold.yaml").read_text())["@context"]

g = rdflib.Dataset()
g.parse(data=json.dumps({"@context": context, **oold_doc}), format="json-ld")

print(f"Graph contains {len(g)} triples.\n")
print(g.serialize(format="turtle"))

In [ ]:
out_ttl = HERE / "output_tensile_test.ttl"
out_ttl.write_text(g.serialize(format="turtle"))
print(f"Written to {out_ttl}")

## Step 4: Validate against SHACL shapes

Two shape files are loaded:

| Shape file | Validates |
|---|---|
| `characterization/step/PMDCo/specs/shape.ttl` | Assay label, specimen input, process conditions |
| `characterization/tensile-test/TTO/specs/shape.ttl` | Tensile test: specimen required, result values and units |

This mirrors the schema inheritance: the base shape validates the inherited
structure, and the specialised shape validates the tensile-test additions.

In [ ]:
shapes_graph = rdflib.Graph()
shapes_graph.parse(str(CHAR_BASE / "specs" / "shape.ttl"))
shapes_graph.parse(str(SCHEMA   / "specs" / "shape.ttl"))

conforms, results_graph, _ = pyshacl.validate(
    g,
    shacl_graph = shapes_graph,
    inference   = "rdfs",
)

print(f"Conforms: {conforms}")
if not conforms:
    SH = rdflib.Namespace("http://www.w3.org/ns/shacl#")
    for result in results_graph.subjects(rdflib.RDF.type, SH.ValidationResult):
        msg  = results_graph.value(result, SH.resultMessage)
        path = results_graph.value(result, SH.resultPath)
        loc  = str(path).rsplit("#", 1)[-1].rsplit("/", 1)[-1] if path else None
        print(f"\n  ✗ {msg}" + (f"\n    property: {loc}" if loc else ""))

## Step 5: Inspect the results

The SPARQL query below extracts the mechanical properties captured in the
graph and prints them as a table.

You do not need to understand SPARQL to use this notebook. Just run the cell
and check that the values match what you entered.

In [ ]:
flat = rdflib.Graph()
for s, p, o, _ in g.quads():
    flat.add((s, p, o))

TTO  = rdflib.Namespace("https://w3id.org/pmd/tto/")
OBI  = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")

proc_iri = next(flat.subjects(rdflib.RDF.type, TTO["TensileTest"]))
print(f"Test IRI : {proc_iri}")
print(f"Label    : {flat.value(proc_iri, rdflib.RDFS.label)}")
print()

SPARQL = """
PREFIX obi:   <http://purl.obolibrary.org/obo/OBI_>
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>
PREFIX tto:   <https://w3id.org/pmd/tto/>
PREFIX uqudt: <https://qudt.org/vocab/unit/>

SELECT ?propertyType ?value ?unit
WHERE {
  ?test  a              tto:TensileTest ;
         obi:0000299    ?result .
  ?result a             ?propertyType ;
          obi:0001937   ?value ;
          iao:0000039   ?unit .
  FILTER(?propertyType != obi:0000299)
}
ORDER BY ?propertyType
"""

rows = list(flat.query(SPARQL))
if rows:
    print(f"{'Property (TTO class)':<45}  {'Value':>8}  Unit")
    print(f"{'':45}  {'':8}  {'':20}")
    for r in rows:
        prop = str(r.propertyType).rsplit("/", 1)[-1].rsplit("#", 1)[-1]
        unit = str(r.unit).rsplit("/", 1)[-1]
        print(f"{prop:<45}  {float(r.value):>8.3f}  {unit}")
else:
    print("No measured properties found in the graph.")

## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the test name, specimen IRI, conditions, and results |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against two SHACL shape files (base + tensile-test) |
| 5 | The measured properties are extracted from the graph to confirm correctness |

To test a different specimen, edit `docs/example.input.json` and re-run all cells.


## Further reading

- [Characterization Step (PMDCo)](../../step/PMDCo/README.md): the base schema this one extends
- [TTO application ontology](https://github.com/materialdigital/application-ontologies/tree/main/tensile_test_ontology_TTO)
- [OO-LD primer](../../../../docs/oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../../docs/schema-format.md): folder structure and naming conventions